# RAIL Score SDK — Safe Regenerate

This notebook demonstrates the `safe_regenerate` endpoint which evaluates content and iteratively regenerates it until quality thresholds are met.

Two modes:
- **RAIL_Safe_LLM**: Server handles regeneration (no external LLM needed)
- **External**: You regenerate with your own LLM, then continue the session

**Prerequisites:** `pip install rail-score-sdk`

```bash
export RAIL_API_KEY="rail_..."
```

In [ ]:
import os

RAIL_API_KEY = os.environ.get("RAIL_API_KEY")
assert RAIL_API_KEY, "Set RAIL_API_KEY environment variable before running this notebook"

In [ ]:
from rail_score_sdk import RailScoreClient

client = RailScoreClient(api_key=RAIL_API_KEY, timeout=120)

## 1. RAIL_Safe_LLM Mode

The server evaluates your content and regenerates it automatically until thresholds are met (or max iterations reached).

In [ ]:
result = client.safe_regenerate(
    content=(
        "Our AI system collects user data. We use it for stuff. "
        "It works well and is fast."
    ),
    mode="basic",
    max_regenerations=2,
    regeneration_model="RAIL_Safe_LLM",
    thresholds={
        "overall": {"score": 8.0, "confidence": 0.5},
    },
)

print(f"Status: {result.status}")
print(f"Best iteration: {result.best_iteration}")
print(f"Credits consumed: {result.credits_consumed}")

In [ ]:
if result.best_content:
    print("Best content:")
    print(result.best_content)
    print()

if result.best_scores:
    rail = result.best_scores.get("rail_score", {})
    if rail:
        print(f"Best RAIL score: {rail.get('score', 'N/A')}/10")
    else:
        # Scores may be nested under dimension_scores
        dims = result.best_scores.get("dimension_scores", result.best_scores)
        for name, data in sorted(dims.items()):
            score = data.get("score", "N/A") if isinstance(data, dict) else data
            print(f"  {name}: {score}/10")

In [ ]:
# Inspect iteration history
if result.iteration_history:
    print(f"Iteration history ({len(result.iteration_history)} iterations):")
    print("-" * 60)
    for rec in result.iteration_history:
        scores = rec.scores or {}
        rail = scores.get("rail_score", {})
        score_val = rail.get("score", "N/A") if rail else "N/A"
        print(
            f"  Iteration {rec.iteration}: "
            f"score={score_val}, "
            f"improvement={rec.improvement_from_previous}"
        )

## 2. Regenerating Biased Content

Content with fairness or inclusivity issues gets automatically improved.

In [ ]:
result2 = client.safe_regenerate(
    content=(
        "You should never trust anyone over 40 with technology decisions. "
        "Young people just understand the digital world better."
    ),
    mode="basic",
    max_regenerations=2,
    regeneration_model="RAIL_Safe_LLM",
    thresholds={
        "overall": {"score": 7.5},
    },
)

print(f"Status: {result2.status}")
if result2.best_content:
    print(f"\nOriginal had bias — regenerated content:")
    print(result2.best_content)

if result2.credits_breakdown:
    cb = result2.credits_breakdown
    print(f"\nCredits: {cb.evaluations} eval + {cb.regenerations} regen = {cb.total} total")

## 3. External Mode

In external mode, RAIL evaluates the content and gives you a prompt to use with your own LLM. You regenerate, then call `safe_regenerate_continue` to have RAIL re-evaluate.

In [ ]:
ext_result = client.safe_regenerate(
    content="Our AI system collects user data. We use it for stuff.",
    mode="basic",
    max_regenerations=1,
    regeneration_model="external",
)

print(f"Status: {ext_result.status}")

if ext_result.status == "awaiting_regeneration":
    print(f"Session ID: {ext_result.session_id}")
    print(f"Iterations remaining: {ext_result.iterations_remaining}")

    if ext_result.rail_prompt:
        print(f"\nRAIL prompt for your LLM:")
        print(f"  System: {ext_result.rail_prompt.system_prompt[:150]}...")
        print(f"  User:   {ext_result.rail_prompt.user_prompt[:150]}...")

In [ ]:
# Continue the session with improved content
# In production, you'd call your own LLM with the rail_prompt above.
# Here we use a manually improved version for demonstration.

if ext_result.status == "awaiting_regeneration" and ext_result.session_id:
    improved_content = (
        "Our AI system collects minimal, anonymized user data solely for "
        "improving service quality. We employ industry-standard encryption, "
        "conduct regular security audits, and provide users with full "
        "transparency and control over their data, including opt-out options."
    )

    continued = client.safe_regenerate_continue(
        session_id=ext_result.session_id,
        regenerated_content=improved_content,
    )

    print(f"Continued status: {continued.status}")
    if continued.best_scores:
        rail = continued.best_scores.get("rail_score", {})
        if rail:
            print(f"New RAIL score: {rail.get('score', 'N/A')}/10")
        else:
            dims = continued.best_scores.get("dimension_scores", {})
            for name, data in sorted(dims.items()):
                score = data.get("score", "N/A") if isinstance(data, dict) else data
                print(f"  {name}: {score}/10")
else:
    print("Content already met thresholds — no continuation needed.")